# Image Analysis
Interactive analysis of images in the lines_added directory

In [ ]:
import os
import re
import hashlib
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Directory paths
BASE_DIR = os.path.dirname(os.path.abspath('.'))
DATA_DIR = os.path.join('.', 'data', 'Sketches')
LINES_ADDED_DIR = os.path.join(DATA_DIR, 'lines_added')
UNRULED_DIR = os.path.join(DATA_DIR, 'Unruled')

## Load or Generate Data

In [ ]:
# Try to load existing CSV, otherwise generate it
csv_path = 'image_analysis.csv'

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"Loaded {len(df)} rows from {csv_path}")
else:
    print("CSV not found. Run analyze_images.py first, or run the generation cell below.")

In [ ]:
# Helper functions
def get_base_name(filename):
    """Extract base name from a filename by removing the suffix like _00001."""
    name_without_ext = os.path.splitext(filename)[0]
    match = re.match(r'(.+)_\d{5}$', name_without_ext)
    if match:
        return match.group(1)
    return name_without_ext

def find_parent_image(lined_filename):
    """Find the corresponding parent image in Unruled directory."""
    base_name = get_base_name(lined_filename)
    for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']:
        parent_path = os.path.join(UNRULED_DIR, base_name + ext)
        if os.path.exists(parent_path):
            return base_name + ext
    return None

def get_file_hash(filepath):
    """Calculate MD5 hash of file contents."""
    hash_md5 = hashlib.md5()
    with open(filepath, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

In [ ]:
# Generate data (run this if CSV doesn't exist)
def generate_dataframe():
    image_files = sorted([
        f for f in os.listdir(LINES_ADDED_DIR)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])
    
    print(f"Analyzing {len(image_files)} images...")
    
    data = []
    for i, filename in enumerate(image_files):
        filepath = os.path.join(LINES_ADDED_DIR, filename)
        
        try:
            file_size = os.path.getsize(filepath)
            parent_image = find_parent_image(filename)
            file_hash = get_file_hash(filepath)
            
            with Image.open(filepath) as img:
                width, height = img.size
                image_format = img.format
            
            data.append({
                'filename': filename,
                'file_size': file_size,
                'parent_image': parent_image,
                'file_hash': file_hash,
                'width': width,
                'height': height,
                'image_format': image_format
            })
            
            if (i + 1) % 1000 == 0:
                print(f"  Processed {i + 1}/{len(image_files)}")
        except Exception as e:
            print(f"Error processing {filename}: {e}")
    
    return pd.DataFrame(data)

# Uncomment to regenerate:
# df = generate_dataframe()
# df.to_csv('image_analysis.csv', index=False)

## Basic Statistics

In [ ]:
print(f"Total images: {len(df)}")
print(f"Unique parent images: {df['parent_image'].nunique()}")
print(f"Images without parent: {df['parent_image'].isna().sum()}")
print(f"Unique file hashes: {df['file_hash'].nunique()}")
df.head()

In [ ]:
df.describe()

## Check for Duplicates

In [ ]:
# Find images with duplicate content (same hash)
duplicate_hashes = df[df.duplicated(subset=['file_hash'], keep=False)]
print(f"Images with duplicate content: {len(duplicate_hashes)}")

if len(duplicate_hashes) > 0:
    print("\nHash counts (top 10):")
    print(duplicate_hashes['file_hash'].value_counts().head(10))

In [ ]:
# Show example duplicates
if len(duplicate_hashes) > 0:
    example_hash = duplicate_hashes['file_hash'].value_counts().index[0]
    print(f"\nExample duplicate files (hash: {example_hash[:16]}...):")
    display(duplicate_hashes[duplicate_hashes['file_hash'] == example_hash])

## Variations per Parent Image

In [ ]:
# Count variations per parent image
variations_per_parent = df.groupby('parent_image').size().reset_index(name='count')
print(f"Parent images with most variations:")
display(variations_per_parent.sort_values('count', ascending=False).head(20))

In [ ]:
# Distribution of variation counts
plt.figure(figsize=(10, 5))
variations_per_parent['count'].hist(bins=30)
plt.xlabel('Number of variations')
plt.ylabel('Number of parent images')
plt.title('Distribution of Variations per Parent Image')
plt.show()

print(f"\nVariation count statistics:")
print(variations_per_parent['count'].describe())

## Image Dimensions

In [ ]:
# Image dimensions distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['width'], bins=30)
axes[0].set_xlabel('Width (pixels)')
axes[0].set_ylabel('Count')
axes[0].set_title('Width Distribution')

axes[1].hist(df['height'], bins=30)
axes[1].set_xlabel('Height (pixels)')
axes[1].set_ylabel('Count')
axes[1].set_title('Height Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Unique dimensions
df['dimensions'] = df['width'].astype(str) + 'x' + df['height'].astype(str)
print("Image dimensions:")
print(df['dimensions'].value_counts())

## Ruled vs Lines Added: Dimension Comparison

In [ ]:
# Load dimension data from Ruled directory
RULED_DIR = os.path.join(DATA_DIR, 'Ruled')

def get_image_dimensions(directory):
    """Get dimensions for all images in a directory."""
    data = []
    image_files = [f for f in os.listdir(directory) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
    for filename in image_files:
        filepath = os.path.join(directory, filename)
        try:
            with Image.open(filepath) as img:
                width, height = img.size
                mode = img.mode
            data.append({
                'filename': filename,
                'width': width,
                'height': height,
                'mode': mode
            })
        except Exception as e:
            print(f"Error processing {filename}: {e}")
    
    return pd.DataFrame(data)

# Get dimensions for both directories
df_ruled = get_image_dimensions(RULED_DIR)
df_lines_added = get_image_dimensions(LINES_ADDED_DIR)

print(f"Ruled images: {len(df_ruled)}")
print(f"Lines Added images: {len(df_lines_added)}")

In [ ]:
# Compare width and height distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Width comparison
axes[0, 0].hist(df_ruled['width'], bins=20, alpha=0.7, label='Ruled', color='blue')
axes[0, 0].set_xlabel('Width (pixels)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Ruled - Width Distribution')
axes[0, 0].legend()

axes[0, 1].hist(df_lines_added['width'], bins=20, alpha=0.7, label='Lines Added', color='orange')
axes[0, 1].set_xlabel('Width (pixels)')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Lines Added - Width Distribution')
axes[0, 1].legend()

# Height comparison
axes[1, 0].hist(df_ruled['height'], bins=20, alpha=0.7, label='Ruled', color='blue')
axes[1, 0].set_xlabel('Height (pixels)')
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('Ruled - Height Distribution')
axes[1, 0].legend()

axes[1, 1].hist(df_lines_added['height'], bins=20, alpha=0.7, label='Lines Added', color='orange')
axes[1, 1].set_xlabel('Height (pixels)')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Lines Added - Height Distribution')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Overlaid comparison histograms
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Width - overlaid
axes[0].hist(df_ruled['width'], bins=30, alpha=0.6, label=f'Ruled (n={len(df_ruled)})', color='blue')
axes[0].hist(df_lines_added['width'], bins=30, alpha=0.6, label=f'Lines Added (n={len(df_lines_added)})', color='orange')
axes[0].set_xlabel('Width (pixels)')
axes[0].set_ylabel('Count')
axes[0].set_title('Width Comparison: Ruled vs Lines Added')
axes[0].legend()

# Height - overlaid
axes[1].hist(df_ruled['height'], bins=30, alpha=0.6, label=f'Ruled (n={len(df_ruled)})', color='blue')
axes[1].hist(df_lines_added['height'], bins=30, alpha=0.6, label=f'Lines Added (n={len(df_lines_added)})', color='orange')
axes[1].set_xlabel('Height (pixels)')
axes[1].set_ylabel('Count')
axes[1].set_title('Height Comparison: Ruled vs Lines Added')
axes[1].legend()

plt.tight_layout()
plt.show()

# Print statistics
print("=" * 60)
print("DIMENSION STATISTICS COMPARISON")
print("=" * 60)
print("\n--- WIDTH ---")
print(f"{'':20} {'Ruled':>12} {'Lines Added':>12}")
print(f"{'Mean':20} {df_ruled['width'].mean():>12.0f} {df_lines_added['width'].mean():>12.0f}")
print(f"{'Median':20} {df_ruled['width'].median():>12.0f} {df_lines_added['width'].median():>12.0f}")
print(f"{'Min':20} {df_ruled['width'].min():>12.0f} {df_lines_added['width'].min():>12.0f}")
print(f"{'Max':20} {df_ruled['width'].max():>12.0f} {df_lines_added['width'].max():>12.0f}")

print("\n--- HEIGHT ---")
print(f"{'':20} {'Ruled':>12} {'Lines Added':>12}")
print(f"{'Mean':20} {df_ruled['height'].mean():>12.0f} {df_lines_added['height'].mean():>12.0f}")
print(f"{'Median':20} {df_ruled['height'].median():>12.0f} {df_lines_added['height'].median():>12.0f}")
print(f"{'Min':20} {df_ruled['height'].min():>12.0f} {df_lines_added['height'].min():>12.0f}")
print(f"{'Max':20} {df_ruled['height'].max():>12.0f} {df_lines_added['height'].max():>12.0f}")

print("\n--- IMAGE MODE ---")
print("Ruled:", df_ruled['mode'].value_counts().to_dict())
print("Lines Added:", df_lines_added['mode'].value_counts().to_dict())

## File Sizes

In [ ]:
# File size distribution
df['file_size_kb'] = df['file_size'] / 1024

plt.figure(figsize=(10, 5))
df['file_size_kb'].hist(bins=50)
plt.xlabel('File Size (KB)')
plt.ylabel('Count')
plt.title('File Size Distribution')
plt.show()

print(f"\nFile size statistics (KB):")
print(df['file_size_kb'].describe())

## Images Without Parent

In [ ]:
# Find images without a matching parent in Unruled
orphans = df[df['parent_image'].isna()]
print(f"Images without parent: {len(orphans)}")

if len(orphans) > 0:
    print("\nOrphan filenames:")
    display(orphans[['filename']].head(20))

## View Sample Images

In [ ]:
def show_image_pair(filename):
    """Display a lined image and its parent side by side."""
    lined_path = os.path.join(LINES_ADDED_DIR, filename)
    parent_name = find_parent_image(filename)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
    # Show lined image
    lined_img = Image.open(lined_path)
    axes[0].imshow(lined_img)
    axes[0].set_title(f'Lined: {filename}')
    axes[0].axis('off')
    
    # Show parent image
    if parent_name:
        parent_path = os.path.join(UNRULED_DIR, parent_name)
        parent_img = Image.open(parent_path)
        axes[1].imshow(parent_img)
        axes[1].set_title(f'Parent: {parent_name}')
    else:
        axes[1].text(0.5, 0.5, 'No parent found', ha='center', va='center')
        axes[1].set_title('Parent: None')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()

# Show a random sample
sample_file = df.sample(1)['filename'].values[0]
show_image_pair(sample_file)